# Generate 200×200 player pictures (Google Colab)

**Important:** Before running any cell, set the runtime to GPU:
- **Runtime → Change runtime type → Hardware accelerator: GPU** (e.g. T4) → Save.
- If you skip this, you'll get "Torch not compiled with CUDA" or runs will be very slow on CPU.

Then run the cells in order. Images will appear below and you can download `player_pictures.zip`.

In [ ]:
# Cell 1: Install packages (run once)
!pip install -q diffusers transformers accelerate Pillow

In [ ]:
# Cell 2: Load a realism-focused model (much better faces than vanilla SD 1.5)
from diffusers import StableDiffusionPipeline
import torch

# Use GPU if available (set Runtime → Change runtime type → GPU in Colab!)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
if device == "cpu":
    print("⚠️ No GPU detected. Enable GPU: Runtime → Change runtime type → GPU, then re-run from Cell 1.")

# Realistic Vision = photorealistic people/faces (SD 1.5 based). Vanilla SD 1.5 often gives distorted faces.
model_id = "SG161222/Realistic_Vision_V5.1_noVAE"
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=dtype,
)
pipe = pipe.to(device)
print(f"Model loaded on {device} ({model_id}).")

In [ ]:
# Cell 3: Generate 200×200 player pictures (varied looks + soccer vibe)
from PIL import Image
import os
import random
import torch

# Variety: pick one per image so each face is different
# Age/build and face shape add a lot of face variability
AGES = [
    "young man", "man in his early 20s", "man in his mid 20s", "man in his late 20s",
    "man in his early 30s", "man in his 30s", "teenager", "man in his 40s",
]
FACE_SHAPES = [
    "oval face", "strong jawline", "round face", "angular face", "square jaw",
    "defined cheekbones", "narrow face", "wide face", "heart-shaped face",
]
# Extra face descriptors – push different bone structure and features
FACIAL_FEATURES = [
    "distinct facial features", "deep set eyes", "high cheekbones", "thick eyebrows",
    "prominent nose", "sharp features", "soft features", "strong brow",
    "defined jaw", "youthful features", "weathered features", "unique face",
]
APPEARANCES = [
    "black man", "white man", "asian man", "latino man", "middle eastern man",
    "south asian man", "mixed race man",
]
HAIRSTYLES = [
    "short hair", "buzz cut", "curly hair", "afro", "long hair", "undercut",
    "braids", "bald", "fade cut", "wavy hair", "dreadlocks", "mohawk", "mullet",
]
# Facial expression per image
EXPRESSIONS = [
    "serious expression", "neutral expression", "slight smile", "smiling", "laughing",
    "focused expression", "confident expression", "calm expression", "intense look",
    "grinning", "stoic expression", "friendly smile",
]
HAIR_COLORS = [
    "black hair", "dark brown hair", "brown hair", "light brown hair", "blonde hair",
    "red hair", "auburn hair", "gray hair", "salt and pepper hair", "dyed black hair",
]
BEARDS = [
    "clean shaven", "clean shaven", "stubble", "stubble", "short beard", "full beard",
    "goatee", "trimmed beard", "light stubble",  # weight clean shaven / stubble a bit more
]
ACCESSORIES = [
    "wearing a bandana", "wearing a headband", "with sweatband",
    "no headwear", "no headwear", "no headwear",  # weight "no accessory" more
]
# Earrings, piercings, tattoos (one per image; "none" weighted so not everyone has them)
PIERCINGS_TATTOOS = [
    "with small earrings", "with earrings", "with ear piercings", "with visible tattoos",
    "with earrings and visible tattoos", "with neck tattoo", "with ear piercing",
    "no visible piercings or tattoos", "no visible piercings or tattoos", "no visible piercings or tattoos",
]
# Different jersey/shirt color per image
JERSEY_COLORS = [
    "red jersey", "blue jersey", "white jersey", "black jersey", "yellow jersey",
    "green jersey", "navy blue jersey", "orange jersey", "purple jersey", "maroon jersey",
    "sky blue jersey", "dark green jersey", "burgundy jersey", "gold jersey", "teal jersey",
]

BASE = "soccer player headshot, athlete, wearing sports jersey, face portrait, head and shoulders, plain gray background, studio lighting, sharp focus, realistic, 8k, photograph"
NEGATIVE = "deformed, disfigured, bad anatomy, bad proportions, extra fingers, extra limbs, mutated, ugly, blurry, low quality, cartoon, painting, drawing, illustration, distorted face, asymmetric eyes, crooked nose, duplicate, mutilated, out of frame, poorly drawn face, suit, tie, formal wear, business attire, shirt and tie, tuxedo, same face, identical face, clone, repeating"

NUM_IMAGES = 5  # change as needed
OUTPUT_DIR = "/content/player_pictures"
STEPS = 40
GUIDANCE = 7.5

os.makedirs(OUTPUT_DIR, exist_ok=True)

for i in range(NUM_IMAGES):
    seed = random.randint(0, 2**32 - 1)  # different seed = different face every time
    generator = torch.Generator(device=pipe.device).manual_seed(seed)

    look = random.choice(APPEARANCES)
    age = random.choice(AGES)
    face_shape = random.choice(FACE_SHAPES)
    face_extra = random.choice(FACIAL_FEATURES)
    hair_style = random.choice(HAIRSTYLES)
    hair_color = random.choice(HAIR_COLORS)
    beard = random.choice(BEARDS)
    expression = random.choice(EXPRESSIONS)
    acc = random.choice(ACCESSORIES)
    extra = random.choice(PIERCINGS_TATTOOS)
    jersey = random.choice(JERSEY_COLORS)
    # Bald doesn't need a hair color
    hair = f"{hair_style}, {hair_color}" if hair_style != "bald" else "bald"
    prompt = f"{BASE}, {look}, {age}, {face_shape}, {face_extra}, {hair}, {beard}, {expression}, {jersey}"
    if acc != "no headwear":
        prompt = f"{prompt}, {acc}"
    if "no visible" not in extra.lower():
        prompt = f"{prompt}, {extra}"
    # Slight guidance jitter (7.2–7.8) so each image follows the prompt a bit differently
    guidance = GUIDANCE + random.uniform(-0.3, 0.3)
    print(f"  [{i+1}] seed={seed} – {age}, {face_extra[:20]}...")
    img = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE,
        width=512,
        height=512,
        num_inference_steps=STEPS,
        guidance_scale=round(guidance, 1),
        generator=generator,
    ).images[0]
    img = img.resize((200, 200), Image.Resampling.LANCZOS)
    path = f"{OUTPUT_DIR}/player_{i:04d}.png"
    img.save(path)
    display(img)
    print(f"Saved {path}")

print(f"Done. All images in {OUTPUT_DIR}")

In [ ]:
# Cell 4: Download all images as a zip
!cd /content && zip -r player_pictures.zip player_pictures/
from google.colab import files
files.download("/content/player_pictures.zip")